<a href="https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2025/0a-agents/hw-03_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq fastmcp

In [2]:
import random
import chat_assistant
from chat_assistant import Tools, ChatAssistant, ChatInterface
import os
import importlib
from groq import Groq
import random
from fastmcp import FastMCP
from fastmcp import Client
import weather_server  # assuming you already ran weather_server.mcp.run()
import mcp_client
import json

In [3]:
known_weather_data = {
    'berlin': 20.0
}

def get_weather(city: str) -> float:
    city = city.strip().lower()

    if city in known_weather_data:
        return known_weather_data[city]

    return round(random.uniform(-5, 35), 1)

#### Q1. Define function description

In [4]:
get_weather_tool = {
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current weather for a given city",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "Name of the city to get the weather for"
        }
      },
      "required": ["city"],
      "additionalProperties": False
    }
  }
}

In [21]:
# Use your Groq client here
from google.colab import userdata
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [5]:
tools = Tools()
tools.add_tool(get_weather, get_weather_tool)

interface = ChatInterface()

developer_prompt = """
You're a helpful weather bot. Use the get_weather tool to answer questions about weather.
"""


chat = ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=interface,
    client=client,
    model="llama3-70b-8192"  # Specify the model to use
)


In [6]:
chat.run()

You:The weather in Berlin?


You:The weather in Kyiv?


You:The weather in Warschaw?


You:stop
Chat ended.


In [10]:
def set_weather(city: str, temp: float) -> None:
    city = city.strip().lower()
    known_weather_data[city] = temp
    return 'OK'

In [11]:
set_weather_tool = {
    "type": "function",
    "function": {
        "name": "set_weather",
        "description": "Add or update the temperature for a city in the weather database.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city to set the temperature for."
                },
                "temp": {
                    "type": "number",
                    "description": "The temperature in Celsius to associate with the city."
                }
            },
            "required": ["city", "temp"],
            "additionalProperties": False
        }
    }
}


In [12]:
tools.add_tool(get_weather, get_weather_tool)
tools.add_tool(set_weather, set_weather_tool)
tools.get_tools()

[{'type': 'function',
  'function': {'name': 'get_weather',
   'description': 'Get the current weather for a given city',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'Name of the city to get the weather for'}},
    'required': ['city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'set_weather',
   'description': 'Add or update the temperature for a city in the weather database.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The name of the city to set the temperature for.'},
     'temp': {'type': 'number',
      'description': 'The temperature in Celsius to associate with the city.'}},
    'required': ['city', 'temp'],
    'additionalProperties': False}}}]

In [13]:

importlib.reload(chat_assistant)


#tools = chat_assistant.Tools()

developer_prompt = """
You are a weather assistant. Use the get weather function to get the weather for a given city and provide the results.
Use the set weather function to set the weather for a given city.
If not city name is provided, return 'No city name provided'.
""".strip()

chat_interface = chat_assistant.ChatInterface()

# client = Groq(api_key=os.getenv("GROQ_API_KEY"))

chat = chat_assistant.ChatAssistant(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client,
    model="llama3-70b-8192"
)

In [14]:
chat.run()

You:The weather in Cairo?


You:The weather in Cairo has been set to 33°C.


You:The weather in Kyiv has been set to 30°C.


You:The weather in Warsaw has been set to 25°C.


You:The weather Cairo?


You:The weather in Berlin?


You:The weather in Gamburg?


You:The weather in Kyiv?


You:The weather in Warsaw?


You:stop
Chat ended.


Function call: get_weather({"city":"Cairo"})
Assistant:
The weather in Cairo is 24.4°C.

Function call: set_weather({"city":"Cairo","temp":33})
Assistant:
The weather in Cairo has been set to 33°C.

Function call: get_weather({"city":"Cairo"})
Assistant:
The weather in Cairo is 33°C.

Chat ended.

In [15]:
known_weather_data

{'berlin': 20.0, 'cairo': 33, 'kyiv': 30, 'warsaw': 25}

#### Q3. Install FastMCP


In [16]:
!pip show fastmcp


Name: fastmcp
Version: 2.11.3
Summary: The fast, Pythonic way to build MCP servers and clients.
Home-page: https://gofastmcp.com
Author: Jeremiah Lowin
Author-email: 
License: 
Location: /usr/local/lib/python3.11/dist-packages
Requires: authlib, cyclopts, exceptiongroup, httpx, mcp, openapi-core, openapi-pydantic, pydantic, pyperclip, python-dotenv, rich
Required-by: 


#### Q4. Simple MCP Server


#### Starting MCP server 'Demo 🚀' with transport server.py:1371                              'stdio'  

#### Q5. Protocol

#### {"jsonrpc":"2.0","id":3,"result":{"content":[{"type":"text","text":"20.0"}],"structuredContent":{"result":20.0},"isError":false}}

#### Q6. Client

In [17]:
async def main():
    async with Client(weather_server.mcp) as mcp_client:
        tools = await mcp_client.list_tools()
        print("Available tools:")
        for tool in tools:
            print(tool)

# Run it in Jupyter
await main()


Available tools:
name='get_weather' title=None description='Retrieves the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to retrieve weather data.\n\nReturns:\n    float: The temperature associated with the city.' inputSchema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'type': 'object'} outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': '_WrappedResult', 'type': 'object', 'x-fastmcp-wrap-result': True} annotations=None meta={'_fastmcp': {'tags': []}}
name='set_weather' title=None description="Sets the temperature for a specified city.\n\nParameters:\n    city (str): The name of the city for which to set the weather data.\n    temp (float): The temperature to associate with the city.\n\nReturns:\n    str: A confirmation string 'OK' indicating successful update." inputSchema={'properties': {'city': {'title': 'City', 'type': 'string'}, 'tem

In [18]:
our_mcp_client = mcp_client.MCPClient(["python", "weather_server.py"])

our_mcp_client.start_server()
our_mcp_client.initialize()
our_mcp_client.initialized()

Started server with command: python weather_server.py
Sending initialize request...
Initialize response: {'protocolVersion': '2024-11-05', 'capabilities': {'experimental': {}, 'prompts': {'listChanged': False}, 'resources': {'subscribe': False, 'listChanged': False}, 'tools': {'listChanged': True}}, 'serverInfo': {'name': 'Demo 🚀', 'version': '1.13.0'}}
Sending initialized notification...
Handshake completed successfully


In [19]:
our_mcp_client.get_tools()
our_mcp_client.call_tool('get_weather', {'city': 'Berlin'})

Retrieving available tools...
Available tools: ['get_weather', 'set_weather', 'get_known_weather_data']
Calling tool 'get_weather' with arguments: {'city': 'Berlin'}


{'content': [{'type': 'text', 'text': '20.0'}],
 'structuredContent': {'result': 20.0},
 'isError': False}

In [23]:
our_mcp_client = mcp_client.MCPClient(["python", "weather_server.py"])

our_mcp_client.start_server()
our_mcp_client.initialize()
our_mcp_client.initialized()

mcp_tools = mcp_client.MCPTools(mcp_client=our_mcp_client)
print("mcp_tools", mcp_tools.get_tools())


developer_prompt = """
You are a helpful assistant that answers in the user's language (English or Russish).
If the request mentions a country instead of a city, use the capital city.
Use get_weather(city) to get temperature.
Use set_weather(city, temp) to set temperature.
Ask user for missing city or temperature if needed.
Always end with a follow-up question.
""".strip()

chat_interface = chat_assistant.ChatInterface()

# client = Groq(api_key=os.getenv("GROQ_API_KEY"))

chat = chat_assistant.ChatAssistant(
    tools=mcp_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    client=client,
    model="llama3-8b-8192"  # Specify the model to use
)

chat.run()

Started server with command: python weather_server.py
Sending initialize request...
Initialize response: {'protocolVersion': '2024-11-05', 'capabilities': {'experimental': {}, 'prompts': {'listChanged': False}, 'resources': {'subscribe': False, 'listChanged': False}, 'tools': {'listChanged': True}}, 'serverInfo': {'name': 'Demo 🚀', 'version': '1.13.0'}}
Sending initialized notification...
Handshake completed successfully
Retrieving available tools...
Available tools: ['get_weather', 'set_weather', 'get_known_weather_data']
mcp_tools [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Retrieves the temperature for a specified city.', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'City'}}, 'required': ['city'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'set_weather', 'description': 'Sets the temperature for a specified city.', 'parameters': {'type': 'object', 'properties': {'city': {'type'

You:The temperature in Cairo?
Calling tool 'get_weather' with arguments: {'city': 'Cairo'}


You:The weather in Warsaw has been successfully set to 25°C.
Calling tool 'set_weather' with arguments: {'city': 'Warsaw', 'temp': 25}


You:Какая температура в Cairo?


You:Какая погода в Гамбурге?
Calling tool 'get_weather' with arguments: {'city': 'Hamburg'}


...

You:Какая температура в Египте и Польше?


You:Какая погода в Германии?


You:Какая столица Германии?


You:Какая погода в Германии?


You:stop
Chat ended.
